In [ ]:
# Figure 5
# Compare the clique and tree-search size approximations to the true twin counts

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

from scipy.stats import kendalltau

%matplotlib inline

In [ ]:
import matplotlib as mpl
title_size = 22
label_size = 22
tick_size = 18
legend_font = 18
labelpad = 3
mpl.rcParams["axes.titlesize"] = title_size
mpl.rcParams["axes.labelpad"] = labelpad
mpl.rcParams["axes.labelsize"] = label_size
mpl.rcParams["axes.titlepad"] = 15
mpl.rcParams["legend.fontsize"] = legend_font
mpl.rcParams["xtick.labelsize"] = tick_size
mpl.rcParams["ytick.labelsize"] = tick_size
mpl.rcParams["font.size"] = 14

In [ ]:
import sys
sys.path.append("../python/")
from distance_stats import read_data
from projection_utils import *

In [ ]:
k = 3
n = 6
results_dir = "../../results/reproduce/exhaustive/"
m_vals = list(range(2, 21))

In [ ]:
# Dictionary from m to the c_k choose m_k approximations for each m and size distribution
approximations = {m:dict() for m in m_vals}
true_vals = {m:dict() for m in m_vals}
all_max_widths = []
all_exact_max_widths = []
all_ms = []
all_twins = []
max_widths = {m:dict() for m in m_vals}
exact_max_widths = {m:dict() for m in m_vals}
total_twins = {m:dict() for m in m_vals}
for m in m_vals:
    print(f"m: {m}")
    # Read the data
    filename = f"n-{n}_m-{m}_k-{k}_non-uniform_exhaustive_projections.csv"
    output_data = read_data(results_dir + filename, xgi_hypergraphs=False)
    
    all_max_widths.extend(output_data["max_width"])
    all_twins.extend(output_data["total_twins"])
    all_ms.extend([m]*len(output_data["total_twins"]))
    # for each projection
    for proj_idx in range(len(output_data["hypergraphs"])):
        max_widths[m][proj_idx] = output_data["max_width"][proj_idx]
        
        total_twins[m][proj_idx] = output_data["total_twins"][proj_idx]
        
        approximations[m][proj_idx] = dict()
        true_vals[m][proj_idx] = dict()
        # Find a 3-uniform hypergraph and compute its projection
        # (not fully necessary, but somewhat of a sense check)
        unif = False
        unif_h = None
        for h in output_data["hypergraphs"][proj_idx]:
            if all([len(he) == 3 for he in h]):
                unif_h = h
                unif = True
                break
                
        if not unif:
            print("Never found a 3-uniform hypergraph...")

        proj = get_projection(n, unif_h)
        
        # Compute C_k of the projection
        cliques = get_cliques(proj)
        emw = get_exact_max_width(proj, cliques=cliques)
        all_exact_max_widths.append(emw)
        exact_max_widths[m][proj_idx] = emw
        # for each size distribution
        for size_dist, info in output_data["dist_results_dict"][proj_idx].items():
            # Compute C_k choose M_k for each size distribution.
            approx = 1
            for he_size, he_num in sorted(size_dist):
                approx *= math.comb(len(cliques[he_size]), he_num)

            approximations[m][proj_idx][size_dist] = approx
            true_vals[m][proj_idx][size_dist] = info["unfiltered"]

In [ ]:
approxs_full = []
twins_full = []
ms_full = []

for m in sorted(list(approximations.keys())):
    for proj_idx in approximations[m]:
        for size_dist in approximations[m][proj_idx]:
            approxs_full.append(approximations[m][proj_idx][size_dist])
            twins_full.append(true_vals[m][proj_idx][size_dist])
            ms_full.append(m)

In [ ]:
## Get an average per projection
approxs = []
twins = []
ms = []

for m in sorted(list(approximations.keys())):
    for proj_idx in approximations[m]:
        approx_dist = np.array([approximations[m][proj_idx][size_dist]
                                for size_dist in approximations[m][proj_idx]])
        approxs.append(approx_dist.mean())
        twins.append(total_twins[m][proj_idx])
        ms.append(m)

max_approx_val = max(approxs)

In [ ]:
# for every m
df_data = {
    "m": [],
    "# Proj.": [],
    "kt-approx-stat": [],
    "kt-approx-pval": [],
    "kt-tree-stat": [],
    "kt-tree-pval": []
}
top_k = 0
xs = []
kts = []
kts_widths = []
for m in sorted(list(approximations.keys())):
    if len(approximations[m]) > 1:
        # Get a list of (proj_idx, avg_approx)
        ranked_by_approx = []
        ranked_by_twins = []
        ranked_by_widths = []
        for proj_idx in approximations[m]:
            approx_dist = np.array([approximations[m][proj_idx][size_dist]
                                    for size_dist in approximations[m][proj_idx]])
            
            ranked_by_approx.append(approx_dist.sum())
            ranked_by_twins.append(total_twins[m][proj_idx])
            ranked_by_widths.append(math.log10(exact_max_widths[m][proj_idx]))

        xs.append(m)
        if top_k > 0:
            kt_approx = kendalltau(ranked_by_twins[0:top_k], ranked_by_approx[0:top_k])
            kts.append(kt_approx.statistic)

            kt_width = kendalltau(ranked_by_twins[0:top_k], ranked_by_widths[0:top_k])
            kts_widths.append(kt_width.statistic)
        else:
            kt_approx = kendalltau(ranked_by_twins, ranked_by_approx)
            kts.append(kt_approx.statistic)
            df_data["m"].append(m)
            df_data["# Proj."].append(len(approximations[m]))
            df_data["kt-approx-stat"].append(kt_approx.statistic)
            df_data["kt-approx-pval"].append(kt_approx.pvalue)
            
            kt_width = kendalltau(ranked_by_twins, ranked_by_widths)
            kts_widths.append(kt_width.statistic)
            df_data["kt-tree-stat"].append(kt_width.statistic)
            df_data["kt-tree-pval"].append(kt_width.pvalue)
        print(f"m: {m}, kendall's tau for approx: {kt_approx.statistic}, p={kt_approx.pvalue}; for widths: {kt_width.statistic}, p={kt_width.pvalue}") 

In [ ]:
text_label_x = 0.09
text_label_y = 0.95

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(13, 9), squeeze=False)# , constrained_layout=True)
#fig.subplots_adjust(wspace=0.2, hspace=0.2)
row = col = 0

xticks=[10**p for p in range(0, 10, 3)]
xticklabels=list(map(lambda p : rf"$10^{{{p}}}$", range(0, 10, 3)))

log_exact_max_widths = list(map(math.log10, all_exact_max_widths))
sc = axs[row][col].scatter(all_twins, log_exact_max_widths, c=all_ms)
axs[row][col].set_ylabel(r"$\prod_{e} {\binom{|\eta_e|}{W_e}}$")
axs[row][col].set_xlabel("Number of Twins")
axs[row][col].set_xscale("log")
yt = list(range(int(min(log_exact_max_widths)), int(max(log_exact_max_widths))+1, 6))
axs[row][col].set_yticks(yt, map(lambda v : f"$10^{{{v}}}$", yt));
axs[row][col].set_xticks(xticks, xticklabels)
axs[row][col].set_xlim(10**-0.5, max(all_twins)*100)
axs[row][col].spines[['right', 'top']].set_visible(False)
axs[row][col].text(text_label_x, text_label_y, "(a)", size=label_size, fontweight="bold", horizontalalignment='center',
     verticalalignment='center', transform=axs[row][col].transAxes)
axs[row][col].set_rasterized(True)

row += 1
max_approx_val = max(approxs)
sc = axs[row][col].scatter(twins_full, approxs_full, c=ms_full)
axs[row][col].plot([min(twins_full), max(approxs_full)], [min(twins_full), max(approxs_full)], color="grey", alpha=0.75)
axs[row][col].set_xlabel("Number of Twins")
axs[row][col].set_title(r"All Combs. of $(\mathcal{C}_k, \mathcal{M}_k)$")
axs[row][col].set_yscale("log")
axs[row][col].set_xscale("log")
axs[row][col].set_ylim((10**-0.5, max_approx_val*100))
axs[row][col].set_xlim((10**-0.5, max_approx_val*100))
axs[row][col].set_xticks(xticks, xticklabels)
axs[row][col].set_yticks([10**p for p in range(0, 13, 2)], map(lambda p : rf"$10^{{{p}}}$", range(0, 13, 2)))
axs[row][col].set_ylabel(r"$\prod_{\mathcal{M}_k>0} {\binom{\mathcal{C}_k}{\mathcal{M}_k}}$")
axs[row][col].spines[['right', 'top']].set_visible(False)
axs[row][col].text(text_label_x, text_label_y, "(b)", size=label_size, fontweight="bold", horizontalalignment='center',
     verticalalignment='center', transform=axs[row][col].transAxes)
axs[row][col].set_rasterized(True)

col += 1
sc = axs[row][col].scatter(twins, approxs, c=ms)
axs[row][col].plot([min(twins_full), max(approxs_full)], [min(twins_full), max(approxs_full)], color="grey", alpha=0.75)
axs[row][col].set_xlabel("Number of Twins")
axs[row][col].set_yscale("log")
axs[row][col].set_xscale("log")
axs[row][col].set_ylim((10**-0.5, max_approx_val*100))
axs[row][col].set_xlim((10**-0.5, max_approx_val*100))
axs[row][col].set_xticks(xticks, xticklabels)
#axs[row][col].set_yticks([])
axs[row][col].set_yticks([10**p for p in range(0, 13, 2)], map(lambda p : rf"$10^{{{p}}}$", range(0, 13, 2)))
axs[row][col].spines[['right', 'top']].set_visible(False)
axs[row][col].set_ylabel(r"$\langle \prod_{\mathcal{M}_k>0} {\binom{\mathcal{C}_k}{\mathcal{M}_k}} \rangle$")
axs[row][col].set_title("Per Proj. Average")
axs[row][col].text(text_label_x, text_label_y, "(c)", size=label_size, fontweight="bold", horizontalalignment='center',
     verticalalignment='center', transform=axs[row][col].transAxes)
cbar = fig.colorbar(sc)
cbar.ax.set_title("$m$")
cbar.ax.set_yticks(range(2, max(m_vals)+1, 2));
axs[row][col].set_rasterized(True)

col = 1
row = 0
axs[row][col].scatter(xs, kts, label="Clique Upper Bound Sum", c = "dodgerblue")
axs[row][col].scatter(xs, kts_widths, marker="^", label="Worst Case Tree Size", c = "darkorange")
axs[row][col].set_title("Rank Correlation w/ Total Twins")
axs[row][col].set_ylabel(r"Kendall's $\tau$")
axs[row][col].set_xlabel(r"$m$")
axs[row][col].set_xticks(xs[::2], xs[::2])
axs[row][col].set_ylim((-1.1, 1.4))
axs[row][col].hlines(1.0, 3, 18, alpha=0.5, ls="--", color="grey")
axs[row][col].legend(frameon=False)
axs[row][col].spines[["right", "top"]].set_visible(False)

axs[row][col].text(text_label_x, text_label_y, "(d)", size=label_size, fontweight="bold", horizontalalignment='center',
     verticalalignment='center', transform=axs[row][col].transAxes)

fig.subplots_adjust(hspace=0.5, wspace=0.35)
fig.savefig("../plots/clique_dist_approximation.pdf", dpi=300, bbox_inches="tight")